## Imports

In [1]:
import pandas as pd

## Loading Data

In [2]:
columns_to_load = [
    'Przystanek nazwa', 'Przystanek numer', 'Lp przystanku', 'Linia', 'Rozkladowy czas odjazdu',
]

df = pd.read_parquet(
    "/teamspace/studios/this_studio/huge_delays_removed_240625.parquet",
    columns=columns_to_load
)

In [3]:
df = df.rename(columns={
    'Przystanek nazwa': 'stop_name',
    'Przystanek numer': 'stop_id',
    'Lp przystanku': 'stop_seq',
    'Linia': 'line',
    'Rozkladowy czas odjazdu' : 'scheduled_departure'
})

df['departure_dow'] = df['scheduled_departure'].dt.dayofweek


df.head()

,stop_name,stop_id,stop_seq,line,scheduled_departure,departure_dow
0,Władysława IV,2169,1,10,2023-01-01 04:11:00,6
1,Nowy Port Góreckiego,208,2,10,2023-01-01 04:12:00,6
2,Marynarki Polskiej,2166,3,10,2023-01-01 04:14:00,6
3,Śnieżna,2164,4,10,2023-01-01 04:16:00,6
4,Polsat Plus Arena Gdańsk,2162,5,10,2023-01-01 04:18:00,6


In [4]:
print(df.shape)

df = df.drop_duplicates(subset=['line', 'stop_seq', 'stop_id', 'stop_name', 'departure_dow'])

print(df.shape)

(74748250, 6)
(102299, 6)


In [5]:
line_stop_to_seq = dict(zip(
    zip(df['line'], df['stop_id']),
    df['stop_seq']
))

In [ ]:
dupes = df.groupby(['line', 'stop_name', 'scheduled_departure'])['stop_seq'].nunique()
non_unique = dupes[dupes > 1]

print("Number of non-unique (line, stop_name) pairs:", len(non_unique))

if not non_unique.empty:
    print("\nExample non-unique pairs:")
    print(non_unique.head(10))
else:
    print("All (line, stop_name) pairs are unique.")


Number of non-unique (line, stop_name) pairs: 410

Example non-unique pairs:
line  stop_name        scheduled_departure
112   Kwiatowa         2023-01-06 08:37:00    2
115   Nad Jarem        2023-01-05 06:27:00    2
      Wyczółkowskiego  2023-01-02 06:26:00    2
                       2023-01-03 06:26:00    2
                       2023-01-04 06:26:00    2
118   Osiedle Jary     2023-01-06 05:16:00    2
12    Strzyża PKM      2024-07-12 03:59:00    2
      Zabornia         2023-01-07 04:46:00    2
124   Wrzeszcz PKP     2023-01-05 04:46:00    2
127   Oliwa PKP        2023-01-01 05:11:00    2
Name: stop_seq, dtype: int64


In [7]:
df = df.sort_values('scheduled_departure')

df_mapping = df.drop_duplicates(subset=['line', 'stop_name', 'scheduled_departure'])[
    ['line', 'stop_name', 'scheduled_departure', 'stop_seq']
]

df_mapping.to_csv("triplet_to_seq.csv", index=False)

print("Saved", len(df_mapping), "unique records to 'triplet_to_seq.csv'")


Saved 101888 unique records to 'triplet_to_seq.csv'
